# 01 - Exploratory Data Analysis (EDA)
## Milk Quality Prediction — Synthetic Dataset

**Tujuan:** Memahami distribusi data, korelasi antar fitur, mendeteksi missing values/outliers, dan menganalisis keseimbangan kelas target (`grade`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'api'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

from train_model import generate_synthetic_data
from ml.pipeline import FEATURE_COLUMNS, NUMERIC_FEATURES, BINARY_FEATURES, ORDINAL_FEATURES

print('Libraries and modules loaded successfully.')

---
## 1. Generate & Load Data

In [ ]:
df = generate_synthetic_data(n=3000)
print(f'\nDataset shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
if missing.sum() == 0:
    print('No missing values — dataset is complete.')
else:
    print('Missing values per column:')
    print(missing[missing > 0])

In [ ]:
df.describe()

---
## 2. Target Distribution (Grade)

In [ ]:
grade_order = ['A', 'B', 'C', 'Reject']
grade_counts = df['grade'].value_counts().reindex(grade_order)
grade_pct = grade_counts / len(df) * 100

print('Grade distribution:')
for g in grade_order:
    print(f'  {g:>6}: {grade_counts[g]:>4} ({grade_pct[g]:.1f}%)')

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

grade_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Grade Counts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].set_xlabel('Grade')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(grade_counts.values):
    axes[0].text(i, v + 15, str(v), ha='center', fontweight='bold')

wedges, texts, autotexts = axes[1].pie(
    grade_counts, labels=grade_counts.index, autopct='%1.1f%%',
    colors=colors, startangle=90, explode=[0.03]*4,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(11)
axes[1].set_title('Grade Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

**Insight:** Synthetic data menampilkan distribusi grade yang tidak terlalu timpang — setiap kelas memiliki representasi yang cukup untuk training model multiclass.

---
## 3. Distribusi Fitur Numerik

In [ ]:
num_cols = NUMERIC_FEATURES
n_cols = 4
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, bins=35, color='#2c81ba', edgecolor='white', linewidth=0.3, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

**Observasi:**
- Sebagian besar fitur numerik mendekati distribusi normal (hasil sampling Gaussian).
- `added_water` memiliki distribusi eksponensial — mayoritas sampel memiliki nilai rendah dengan ekor kanan panjang.
- Beberapa fitur (seperti `density`, `freezing_point`) memiliki rentang sangat sempit, yang berarti scaling akan penting.

---
## 4. Distribusi Fitur Kategorikal & Ordinal

In [ ]:
cat_features = BINARY_FEATURES + ORDINAL_FEATURES

fig, axes = plt.subplots(1, len(cat_features), figsize=(5 * len(cat_features), 4))
if len(cat_features) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_features):
    vc = df[col].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color='#2c81ba', edgecolor='white', linewidth=0.8)
    ax.set_title(f'{col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for x, v in zip(vc.index.astype(str), vc.values):
        ax.text(x, v + max(vc.values)*0.02, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Correlation Heatmap

In [ ]:
grade_map = {'A': 0, 'B': 1, 'C': 2, 'Reject': 3}
df_corr = df.copy()
df_corr['grade_encoded'] = df['grade'].map(grade_map)

corr_cols = NUMERIC_FEATURES + ['grade_encoded']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Numeric Features vs Grade', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretasi:**
- `total_solid` sangat berkorelasi positif dengan `fat` + `snf` + `salt` (karena total_solid = jumlah ketiganya).
- Fitur dengan korelasi tertinggi terhadap `grade_encoded` (lower is better): `alcohol_test`, `peroxide_test`, `added_water`, dan `ph`.
- Multikolinearitas antar fitur tidak terlalu parah, sehingga sebagian besar model akan stabil.

---
## 6. Boxplots by Grade

In [ ]:
key_features = ['ph', 'fat', 'snf', 'temperature', 'added_water', 'protein', 'density', 'freezing_point']
n_rows = 2
n_cols = 4

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 10))
axes = axes.flatten()

palette_colors = {'A': '#2ecc71', 'B': '#3498db', 'C': '#f39c12', 'Reject': '#e74c3c'}

for i, col in enumerate(key_features):
    sns.boxplot(
        data=df, x='grade', y=col, ax=axes[i],
        order=grade_order, palette=palette_colors, linewidth=1.2,
        width=0.6, fliersize=3
    )
    axes[i].set_title(f'{col} vs Grade', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Grade')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

**Pola yang terlihat:**
- **pH**: Grade A cenderung memiliki pH sekitar 6.6-6.8. Grade Reject memiliki variasi lebih besar.
- **added_water**: Grade A memiliki added_water sangat rendah. Grade C & Reject cenderung lebih tinggi.
- **fat, snf, protein**: Nilai lebih tinggi berkorelasi dengan grade yang lebih baik (A > B > C > Reject).
- **temperature**: Grade A cenderung memiliki suhu lebih rendah (≤6°C).
- **freezing_point**: Grade A memiliki freezing point lebih rendah (lebih negatif).

---
## 7. Pairplot of Selected Features

In [ ]:
pair_cols = ['ph', 'fat', 'added_water', 'temperature', 'grade']
sns.pairplot(df[pair_cols], hue='grade', palette=palette_colors, diag_kind='kde', height=2.5)
plt.suptitle('Pairplot — Selected Features by Grade', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
## 8. Key Insights Summary

### Dataset Overview
- **3000 samples** synthetic dengan **16 fitur** + 1 target (`grade`).
- Tidak ada missing values.
- Tidak ada duplikat signifikan.

### Target
- 4 kelas ordinal: **A** (terbaik), **B**, **C**, **Reject**.
- Distribusi cukup seimbang — setiap kelas ≥15% dari total.

### Fitur Paling Diskriminatif
1. **alcohol_test / peroxide_test** — jika positif → langsung Reject.
2. **added_water** — makin tinggi makin buruk.
3. **ph** — zona ideal 6.6-6.8.
4. **fat, snf, protein** — komposisi nutrisi.
5. **temperature** — suhu penyimpanan dingin lebih baik.

### Action Items untuk Preprocessing
- Scaling diperlukan karena rentang fitur sangat bervariasi.
- Tidak perlu encoding khusus untuk `alcohol_test`/`peroxide_test` (sudah biner 0/1).
- Stratified split diperlukan untuk menjaga distribusi grade.
- Pipeline dari `api.ml.pipeline` sudah mencakup imputasi, reindex kolom, dan scaling.